<a href="https://colab.research.google.com/github/Suim00/hdac-knhanes-skipbreakfast-obesity/blob/main/code/10_table1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10주차 Table 1 (9팀)
국민건강영양조사 제9기(2022-2024) 중 2024년 자료. 아침 결식(이분형) 기준 Table 1, 흐름도, 비가중 대 가중 비교.

## 1. 내 Google Drive 연결 및 자료 읽기

In [1]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/hdac_data/HN24_ALL(SAS)'
DATA_FILE = f'{DATA_DIR}/hn24_all.sas7bdat'

Mounted at /content/drive


In [2]:
!pip -q install pyreadstat
import pyreadstat, numpy as np, pandas as pd

need = ['age', 'sex', 'HE_prg', 'L_BR_FQ', 'HE_ht', 'HE_wt', 'HE_BMI', 'HE_obe',
        'wt_tot', 'kstrata', 'psu', 'town_t', 'educ', 'sm_presnt', 'dr_month', 'N_DIET', 'N_EN']

_, meta0 = pyreadstat.read_sas7bdat(DATA_FILE, metadataonly=True)
lower_map = {c.lower(): c for c in meta0.column_names}
present = {v: lower_map.get(v.lower()) for v in need}
assert all(present.values()), [v for v, a in present.items() if a is None]

df, meta = pyreadstat.read_sas7bdat(DATA_FILE, usecols=list(present.values()))
df = df.rename(columns={a: v for v, a in present.items()})
print(df.shape)

(6997, 17)


## 2. 분석용 변수 만들기
원본 행은 지우지 않고 표시변수만 만든다.

- 노출: `L_BR_FQ` 3·4 = 결식(1), 1·2 = 아침 식사(0), 9·결측 = 결측 (이용지침서 218쪽)
- 결과: `HE_obe` 4~6 = 비만(BMI 25 이상) (이용지침서 179쪽)
- 공변량: `educ` 88·99, `N_DIET` 9는 결측 처리 (이용지침서 134·232쪽)
- 교육수준 묶음(`edu4`)은 팀이 확정한 기준으로 바꿀 수 있다.

In [3]:
df['breakfast_skip'] = df['L_BR_FQ'].map({1: 0, 2: 0, 3: 1, 4: 1})
df['obese'] = np.where(df['HE_obe'].isna(), np.nan, (df['HE_obe'] >= 4).astype(float))
df['educ_c'] = df['educ'].where(~df['educ'].isin([88, 99]))
df['edu4'] = df['educ_c'].map({1: 1, 2: 1, 3: 1, 4: 2, 5: 3, 6: 4, 7: 4, 8: 4})   # 묶음은 팀이 확정
df['N_DIET_c'] = df['N_DIET'].where(~df['N_DIET'].isin([9]))
df['clu'] = df['kstrata'].astype(str) + '_' + df['psu'].astype(str)   # 집락 식별자 (층 + 조사구)

## 3. B-1 흐름도 (분석 대상 표시변수 방식)
행을 지우지 않고 `in_study`(분석 대상), `design_ok`(설계 변수 유효) 표시변수로 단계별 인원과 제외 사유를 정리한다.

In [4]:
conds = [
    ('1. 만 19세 이상', df['age'] >= 19, '연구 대상 아님(비해당)'),
    ('2. 임신 중 아님', df['HE_prg'] != 1, '제외 기준(임신)'),
    ('3. 아침식사 빈도 응답', df['L_BR_FQ'].isin([1, 2, 3, 4]), '결측(노출 정의 불가)'),
    ('4. 신장·체중·BMI 값 있음', df[['HE_ht', 'HE_wt', 'HE_BMI']].notna().all(axis=1),
     '결측(BMI 이상값 기준은 적용하지 않음)'),
    ('5. 가중치·층·집락 유효',
     df['wt_tot'].notna() & (df['wt_tot'] > 0) & df['kstrata'].notna() & df['psu'].notna(),
     '설계상 제외'),
]

cur = pd.Series(True, index=df.index)
rows = [{'단계': '0. 전체 조사 참여자', '제외 인원': 0, '남은 인원': int(cur.sum()), '사유': '출발 대상'}]
for name, cond, why in conds:
    before = int(cur.sum())
    cur = cur & cond
    rows.append({'단계': name, '제외 인원': before - int(cur.sum()), '남은 인원': int(cur.sum()), '사유': why})

df['in_study'] = cur.astype(int)      # 분석 대상 표시변수
df['design_ok'] = conds[-1][1]        # 설계 변수 유효 표시변수
flow = pd.DataFrame(rows)
print(flow.to_string(index=False))

base = (df['age'] >= 19) & (df['HE_prg'] != 1) & df['L_BR_FQ'].isin([1, 2, 3, 4])
print()
print('4단계 변수별 결측:', df.loc[base, ['HE_ht', 'HE_wt', 'HE_BMI']].isna().sum().to_dict())

               단계  제외 인원  남은 인원                      사유
     0. 전체 조사 참여자      0   6997                   출발 대상
      1. 만 19세 이상    964   6033           연구 대상 아님(비해당)
       2. 임신 중 아님     16   6017               제외 기준(임신)
    3. 아침식사 빈도 응답    174   5843            결측(노출 정의 불가)
4. 신장·체중·BMI 값 있음     94   5749 결측(BMI 이상값 기준은 적용하지 않음)
   5. 가중치·층·집락 유효      0   5749                  설계상 제외

4단계 변수별 결측: {'HE_ht': 94, 'HE_wt': 23, 'HE_BMI': 94}


## 4. 설계변수 점검
가중치 `wt_tot`, 층 `kstrata`, 집락 `psu`가 이 분석에 맞는 설계변수인지 확인한다. 분석 대상은 자료를 잘라내지 않고 도메인(부분모집단)으로 처리한다.

In [5]:
d = df[df['design_ok']].copy()        # 설계 정보가 있는 전체 (도메인 분석용)
print('설계 자료 n:', len(d), '/ 분석 대상(도메인) n:', int(d['in_study'].sum()))
print('층 수:', d['kstrata'].nunique(), '/ 집락(PSU) 수:', d['clu'].nunique())
psu_per_strata = d.groupby('kstrata')['clu'].nunique()
print('층당 PSU 수 최소:', int(psu_per_strata.min()), '(2 이상이어야 분산 추정 가능)')
print('가중치 합(추정 모집단, 명):', round(d['wt_tot'].sum()))

설계 자료 n: 6805 / 분석 대상(도메인) n: 5749
층 수: 27 / 집락(PSU) 수: 192
층당 PSU 수 최소: 2 (2 이상이어야 분산 추정 가능)
가중치 합(추정 모집단, 명): 51522035


## 5. 함수 (복합표본 평균·SE, 가중 분위수)

In [6]:
def _var_ratio(d, u):
    t = pd.DataFrame({'h': d['kstrata'].values, 'c': d['clu'].values, 'u': u})
    uc = t.groupby(['h', 'c'])['u'].sum().reset_index()
    var = 0.0
    for h, g in uc.groupby('h'):
        n = len(g)
        if n > 1:
            var += n / (n - 1) * ((g['u'] - g['u'].mean()) ** 2).sum()
    return var

def svy_mean(d, y, dom):
    """도메인(dom=True)에서 y의 가중 평균과 SE. y가 결측인 행은 도메인에서 제외."""
    ok = (dom & d[y].notna()).values
    w = d['wt_tot'].values
    yy = d[y].fillna(0).values.astype(float)
    W = (w * ok).sum()
    m = (w * ok * yy).sum() / W
    u = w * ok * (yy - m) / W
    return m, np.sqrt(_var_ratio(d, u)), int(ok.sum())

def wquant(x, w, qs=(0.25, 0.5, 0.75)):
    o = np.argsort(x)
    x = np.asarray(x)[o]
    w = np.asarray(w)[o]
    cw = np.cumsum(w) / w.sum()
    return [x[np.searchsorted(cw, q)] for q in qs]

## 6. B-2 Table 1 (가중)
열은 전체 / 비결식군 / 결식군이며 열 머리에 비가중 n을 표시한다. 범주형은 n (가중 %), 연속형은 가중 평균(SE) 또는 가중 중앙값[IQR]이다. 공변량 결측은 제외하지 않고 변수별 '결측 n'으로 적는다.

In [7]:
groups = {
    '전체': d['in_study'] == 1,
    '비결식군': (d['in_study'] == 1) & (d['breakfast_skip'] == 0),
    '결식군': (d['in_study'] == 1) & (d['breakfast_skip'] == 1),
}

cont = [('나이(세), 평균(SE)', 'age', 'mean'),
        ('BMI(kg/m²), 평균(SE)', 'HE_BMI', 'mean'),
        ('총에너지 섭취(kcal), 중앙값[IQR]', 'N_EN', 'median')]
cat = [
    ('성별', 'sex', {1: '남자', 2: '여자'}),
    ('거주지역', 'town_t', {1: '동', 2: '읍·면'}),
    ('교육수준', 'edu4', {1: '초졸 이하', 2: '중졸', 3: '고졸', 4: '대졸 이상'}),
    ('현재흡연', 'sm_presnt', {0: '아니오', 1: '예'}),
    ('월간음주', 'dr_month', {0: '월 1회 미만', 1: '월 1회 이상'}),
    ('식사요법', 'N_DIET_c', {1: '예', 2: '아니오'}),
    ('비만(BMI 25 이상)', 'obese', {0: '아니오', 1: '예'}),
]

def fmt_cont(y, kind, dom):
    miss = int((dom & d[y].isna()).sum())
    if kind == 'mean':
        m, se, n = svy_mean(d, y, dom)
        return f'{m:.1f} ({se:.2f})', miss
    sel = dom & d[y].notna()
    q = wquant(d.loc[sel, y].values, d.loc[sel, 'wt_tot'].values)
    return f'{q[1]:.0f} [{q[0]:.0f}–{q[2]:.0f}]', miss

rows = []
head = {'변수': '', '구분': ''}
for g, dom in groups.items():
    head[g] = f'n = {int(dom.sum()):,}'          # 비가중 n
rows.append(head)

for label, y, kind in cont:
    r, ms = {'변수': label, '구분': ''}, {}
    for g, dom in groups.items():
        r[g], ms[g] = fmt_cont(y, kind, dom)
    rows.append(r)
    rows.append({'변수': '  결측 n', '구분': '', **ms})

for label, x, mp in cat:
    rows.append({'변수': label, '구분': '', **{g: '' for g in groups}})
    pct_sum = {g: 0.0 for g in groups}
    for code_, name in mp.items():
        r = {'변수': '', '구분': name}
        tmp = d.assign(_ind=(d[x] == code_).astype(float).where(d[x].notna()))
        for g, dom in groups.items():
            m, se, n = svy_mean(tmp, '_ind', dom)
            cnt = int(((d[x] == code_) & dom).sum())
            r[g] = f'{cnt:,} ({m * 100:.1f})'
            pct_sum[g] += m * 100
        rows.append(r)
    assert all(abs(v - 100) < 0.01 for v in pct_sum.values()), (x, pct_sum)   # 범주 비율 합 100% 확인
    rows.append({'변수': '  결측 n', '구분': '',
                 **{g: int((dom & d[x].isna()).sum()) for g, dom in groups.items()}})

table1 = pd.DataFrame(rows)
print(table1.to_string(index=False))

                     변수      구분               전체             비결식군              결식군
                                       n = 5,749        n = 4,063        n = 1,686
          나이(세), 평균(SE)              49.9 (0.49)      55.4 (0.48)      39.4 (0.45)
                   결측 n                        0                0                0
     BMI(kg/m²), 평균(SE)              24.3 (0.07)      24.2 (0.08)      24.3 (0.12)
                   결측 n                        0                0                0
총에너지 섭취(kcal), 중앙값[IQR]         1735 [1284–2293] 1741 [1312–2284] 1722 [1227–2310]
                   결측 n                        2                2                0
                     성별                                                           
                             남자     2,474 (50.2)     1,735 (49.0)       739 (52.5)
                             여자     3,275 (49.8)     2,328 (51.0)       947 (47.5)
                   결측 n                        0                0                0
    

In [8]:
# 문서(docs/10_table1.md)에 붙여 넣을 마크다운 표
print(table1.to_markdown(index=False))

| 변수                             | 구분        | 전체             | 비결식군         | 결식군           |
|:---------------------------------|:------------|:-----------------|:-----------------|:-----------------|
|                                  |             | n = 5,749        | n = 4,063        | n = 1,686        |
| 나이(세), 평균(SE)               |             | 49.9 (0.49)      | 55.4 (0.48)      | 39.4 (0.45)      |
| 결측 n                           |             | 0                | 0                | 0                |
| BMI(kg/m²), 평균(SE)             |             | 24.3 (0.07)      | 24.2 (0.08)      | 24.3 (0.12)      |
| 결측 n                           |             | 0                | 0                | 0                |
| 총에너지 섭취(kcal), 중앙값[IQR] |             | 1735 [1284–2293] | 1741 [1312–2284] | 1722 [1227–2310] |
| 결측 n                           |             | 2                | 2                | 0                |
| 성별                             |             |               

### 각주 (문서에 함께 적기)
n은 비가중 표본 수이고 %는 가중치를 적용한 비율(해당 변수 값이 있는 대상자 기준)이다. 연속형은 가중 평균(표준오차, SE) 또는 가중 중앙값[사분위범위]이며, 총에너지 섭취는 분포가 한쪽으로 치우쳐 중앙값[IQR]으로 제시하였다. 가중치 `wt_tot`, 층 `kstrata`, 집락 `psu`를 적용하고(이용지침서 93·94쪽, 가중치 선택 24쪽·표 6-2 28쪽, 복합표본 분석 35쪽) 표준오차는 Taylor 선형화로 추정하였다. 분석 대상(노출·결과가 유효한 5,749명)은 자료를 삭제하지 않고 도메인으로 지정하였다. 공변량 결측은 제외하지 않고 변수별 '결측 n'으로 표시하였으며, 교육수준 99(모름·무응답)와 식사요법 9는 결측으로 처리하였다. 나이는 80세 이상이 80으로 코딩되어 있다. 자료: 국민건강영양조사 제9기(2022-2024) 중 2024년.

### 해석 주의
Table 1의 비만 비율과 공변량 분포 차이는 기술통계이며, 연령·생활습관 차이가 교란변수인지는 11주차 DAG에서 판단한다. 아침식사 빈도별 비만 비율은 예상과 다르더라도 결과 그대로 제시한다.

## 7. B-3 비가중 대 가중 비교
같은 대상(분석 대상 5,749명)과 같은 정의로 비가중 비율·평균과 가중 비율·평균을 비교하고, 차이가 큰 변수 3개를 고른다.
비교 기준: 상대차이(%) = |가중 − 비가중| / 비가중 × 100

In [9]:
dom_all = groups['전체']
cmp_cat = cat + [('아침 결식', 'breakfast_skip', {1: '결식'})]
out = []

for label, y, kind in cont[:2]:
    u = d.loc[dom_all, y].mean()
    w_ = svy_mean(d, y, dom_all)[0]
    out.append((label, u, w_))

for label, x, mp in cmp_cat:
    for code_, name in mp.items():
        tmp = d.assign(_ind=(d[x] == code_).astype(float).where(d[x].notna()))
        u = (d.loc[dom_all, x].dropna() == code_).mean() * 100
        w_ = svy_mean(tmp, '_ind', dom_all)[0] * 100
        out.append((f'{label}: {name}', u, w_))

cmp = pd.DataFrame(out, columns=['항목', '비가중', '가중'])
cmp['차이(가중-비가중)'] = cmp['가중'] - cmp['비가중']
cmp['상대차이(%)'] = cmp['차이(가중-비가중)'].abs() / cmp['비가중'] * 100
cmp = cmp.sort_values('상대차이(%)', ascending=False).round(2)
print(cmp.to_string(index=False))
print()
print('차이가 큰 3개:')
print(cmp.head(3).to_string(index=False))

                항목   비가중    가중  차이(가중-비가중)  상대차이(%)
       교육수준: 초졸 이하 13.91  9.37       -4.54    32.62
          교육수준: 중졸 10.77  8.39       -2.38    22.10
         아침 결식: 결식 29.33 34.85        5.52    18.83
         거주지역: 읍·면 20.07 16.48       -3.59    17.89
       교육수준: 대졸 이상 48.24 56.64        8.40    17.42
            성별: 남자 43.03 50.24        7.21    16.75
           현재흡연: 예 13.57 15.74        2.17    16.01
            성별: 여자 56.97 49.76       -7.21    12.65
     월간음주: 월 1회 미만 48.65 43.84       -4.81     9.89
     월간음주: 월 1회 이상 51.35 56.16        4.81     9.37
     나이(세), 평균(SE) 53.91 49.86       -4.05     7.52
          교육수준: 고졸 27.08 25.60       -1.49     5.49
           거주지역: 동 79.93 83.52        3.59     4.49
  비만(BMI 25 이상): 예 36.51 37.71        1.20     3.29
         현재흡연: 아니오 86.43 84.26       -2.17     2.51
비만(BMI 25 이상): 아니오 63.49 62.29       -1.20     1.89
           식사요법: 예 32.40 32.08       -0.32     0.99
BMI(kg/m²), 평균(SE) 24.13 24.26        0.13     0.53
         식사요

### 비가중 표가 모집단을 반영하지 못하는 이유 (문서에 함께 적기)
가중치는 표본설계 시점과 조사 시점 간 포함오차, 불균등 추출률, 조사 미참여자의 무응답오차 등을 보정하며(이용지침서 23쪽), 설계가중치(추출률의 역수), 무응답 조정, 사후층화 과정을 거쳐 산출된다(25쪽). 또한 표본은 2단계 층화집락표본으로 추출되었으므로 분석에 복합표본 요소를 반영해야 한다(35쪽). 따라서 비가중 표는 모집단의 분포를 반영하지 못한다.